In [2]:
import pandas as pd
import altair as alt

df = pd.read_csv('videogamesales.csv')
df = df.dropna(subset=['Year'])
df['Publisher'] = df['Publisher'].fillna('Unknown')
df['Year'] = df['Year'].astype(int)
df = df[(df['Year'] >= 1980) & (df['Year'] <= 2016)]

genre_yearly = (
    df.groupby(['Year', 'Genre'])['Global_Sales']
    .sum().reset_index()
    .rename(columns={'Global_Sales': 'Total_Global_Sales'})
)
genre_yearly['Total_Global_Sales'] = genre_yearly['Total_Global_Sales'].round(2)

total_yearly = (
    df.groupby('Year')['Global_Sales']
    .sum().reset_index()
    .rename(columns={'Global_Sales': 'Total_Global_Sales'})
)
total_yearly['Genre'] = 'All Genres'
total_yearly['Total_Global_Sales'] = total_yearly['Total_Global_Sales'].round(2)

combined = pd.concat([total_yearly, genre_yearly], ignore_index=True)

genres = ['All Genres'] + sorted(df['Genre'].unique().tolist())

genre_select = alt.binding_select(options=genres, name='Genre: ')
genre_param = alt.param(name='genre_select', bind=genre_select, value='All Genres')

hover = alt.selection_point(on='mouseover', nearest=True, fields=['Year'])

line = (
    alt.Chart(combined)
    .mark_line(point=True, color='#4C78A8', strokeWidth=2.5)
    .encode(
        x=alt.X('Year:O', title='Year', axis=alt.Axis(labelAngle=-45)),
        y=alt.Y('Total_Global_Sales:Q', title='Total Global Sales (millions)', scale=alt.Scale(zero=False)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Genre:N', title='Genre'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter('datum.Genre === genre_select')
    .properties(width=700, height=350, title='Global Video Game Sales Over the Years')
)

rule = (
    alt.Chart(combined)
    .mark_rule(color='gray', strokeDash=[4, 4])
    .encode(
        x='Year:O',
        opacity=alt.condition(hover, alt.value(0.8), alt.value(0)),
        tooltip=[
            alt.Tooltip('Year:O', title='Year'),
            alt.Tooltip('Genre:N', title='Genre'),
            alt.Tooltip('Total_Global_Sales:Q', title='Global Sales (M)', format='.1f')
        ]
    )
    .transform_filter('datum.Genre === genre_select')
    .add_params(hover)
)

chart = (
    (line + rule)
    .add_params(genre_param)
    .configure_view(strokeWidth=0)
)

chart.save('line_chart_export.html')
chart

alt.LayerChart(...)